In [3]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")
df = pd.read_csv(os.path.join(path, "tmdb_5000_movies.csv"))

Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.


In [6]:
df = df[['title', 'overview']]
df['overview'] = df['overview'].fillna("")

print(df.head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


In [7]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df['overview'])

In [8]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [9]:
# Recommendation Function
def recommend_movies(title, top_n=5):
    if title not in indices:
        return f"Movie '{title}' not found in dataset."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the input movie itself
    sim_scores = sim_scores[1: top_n + 1]

    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices].values

In [15]:
movie = "The Dark Knight Rises"
print(f"Recommendations for {movie}:\n")
print(recommend_movies(movie))

Recommendations for The Dark Knight Rises:

['Batman Forever' 'The Dark Knight' 'Batman' 'Batman Returns' 'Slow Burn']
